In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

# ==============================
# GPU CONFIG
# ==============================
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        print("GPU Enabled")
    except:
        print("GPU Error")

# ==============================
# DATASET PATH
# ==============================
data_dir = r"Hair Diseases - Final"

train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")
test_dir  = os.path.join(data_dir, "test")

# ==============================
# DATA AUGMENTATION
# ==============================
train_aug = ImageDataGenerator(
    rescale=1/255.0,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2]
)

val_aug = ImageDataGenerator(rescale=1/255.0)

test_aug = ImageDataGenerator(rescale=1/255.0)

# ==============================
# DATA LOADERS
# ==============================
train_gen = train_aug.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

val_gen = val_aug.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

test_gen = test_aug.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_gen.num_classes
print("Classes:", train_gen.class_indices)

# ==============================
# RESNET50 MODEL
# ==============================
base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base.trainable = True  # fine-tuning

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(1e-4),
    metrics=["accuracy"]
)

model.summary()

# ==============================
# CALLBACKS (BEST MODEL SAVE)
# ==============================
save_path = r"D:\disease prediction model\required_files\other_files\best_model_tf.h5"

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=save_path,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max",
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.1,
    patience=5,
    verbose=1
)

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

# ==============================
# TRAINING
# ==============================
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30,
    callbacks=[checkpoint_cb, reduce_lr, early_stop]
)

print("Training finished!")
print(f"Best model saved to: {save_path}")

# ==============================
# TEST ACCURACY
# ==============================
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

# Folder containing: train / val / test
data_dir = r"Hair Diseases - Final"

train_dir = os.path.join(data_dir, "train")
val_dir   = os.path.join(data_dir, "val")
test_dir  = os.path.join(data_dir, "test")

# ------------------------------------
# DATA AUGMENTATION
# ------------------------------------
train_aug = ImageDataGenerator(
    rescale=1/255.0,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_aug = ImageDataGenerator(rescale=1/255.0)
test_aug = ImageDataGenerator(rescale=1/255.0)

# ------------------------------------
# DATA LOADERS
# ------------------------------------
train_gen = train_aug.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

val_gen = val_aug.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical"
)

test_gen = test_aug.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    shuffle=False
)

num_classes = train_gen.num_classes
print("Detected Classes:", train_gen.class_indices)

# ------------------------------------
# MODEL (ResNet50)
# ------------------------------------
base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base.trainable = True  # train all layers

x = base.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base.input, outputs=output)

model.compile(
    loss="categorical_crossentropy",
    optimizer=Adam(1e-4),
    metrics=["accuracy"]
)

model.summary()

# ------------------------------------
# TRAINING (Start always from Epoch 1)
# ------------------------------------
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=30
)

print("Training Completed!")

# ------------------------------------
# TEST ACCURACY
# ------------------------------------
test_loss, test_acc = model.evaluate(test_gen)
print(f"Test Accuracy: {test_acc:.4f}")


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from PIL import Image


**Safe Dataset Loader**

In [ ]:
class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        try:
            return super(SafeImageFolder, self).__getitem__(index)
        except (FileNotFoundError, OSError, Image.DecompressionBombError):
            print(f"⚠️ Skipping corrupted/missing file: {self.samples[index][0]}")
            new_index = (index + 1) % len(self.samples)
            return self.__getitem__(new_index)


**Device Setup (GPU/CPU)**

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True


**Dataset Path**

In [ ]:
data_dir = r"Hair Diseases - Final"


**Data Augmentation + Normalization**

In [ ]:
data_transforms = {
    "train": transforms.Compose([...]),
    "val": transforms.Compose([...]),
    "test": transforms.Compose([...]),
}


**Datasets & DataLoaders**

In [ ]:
image_datasets = { ... }

dataloaders = { ... }

dataset_sizes = { ... }
class_names = image_datasets["train"].classes
num_classes = len(class_names)


**Model Setup (Transfer Learning)**

In [ ]:
model = models.resnet50(pretrained=True)
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.fc.in_features, num_classes)
)


**Loss Function, Optimizer, Scheduler**

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
scaler = torch.cuda.amp.GradScaler()


**Resume Training from Checkpoint**

In [ ]:
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint["model_state"])


**Training + Validation Loop**

In [ ]:
for epoch in range(start_epoch, num_epochs):
    for phase in ["train", "val"]:
        model.train() / model.eval()
        for inputs, labels in dataloaders[phase]:
            ...


**Saving Best Model**

In [ ]:
if phase == "val" and epoch_acc > best_acc:
    torch.save(model.state_dict(), "best_model.pth")


**Completing Training**

In [ ]:
print("Training complete!")
print(f"Best validation accuracy: {best_acc:.4f}")


****

****